
# 練習問題: ループを複数スレッドで分担する

## 目標

`#pragma omp parallel for` (Fortran では `!$omp parallel do` ... `!$omp end parallel do`) を1つ挿入するだけで, ループの繰り返しを複数のスレッドに分担させられることを体験する.

## 課題

`parallel_for.cpp` (または `parallel_for.f90`) は, 配列 `a[i]` に独立な計算結果を書き込むループを, 現状では1スレッドで実行している.
各繰り返しがどのスレッドで実行されたかも表示する.

コメント `TODO` の指示に従って **OpenMP の指示行を1つ追加** し, ループの繰り返しが複数のスレッドに分担されるようにせよ.

- C++: `for` ループの直前に `#pragma omp parallel for` を1行加える.
- Fortran: `do` ループを `!$omp parallel do` と `!$omp end parallel do` で囲む.

それ以外のコードを変更する必要はない.

## コンパイルと実行

```
# C++
nvc++ -fast -mp=multicore parallel_for.cpp -o parallel_for.exe

# Fortran
nvfortran -fast -mp=multicore parallel_for.f90 -o parallel_for.exe
```

```
OMP_NUM_THREADS=4 ./parallel_for.exe
```

## 期待される結果

各繰り返し `i` が, いずれかのスレッドによって実行される. 1スレッドだけが全繰り返しを実行するのではなく, 複数のスレッド番号が現れる (表示の順番は実行ごとに入れ替わってよい). 例:

```
a[0] = 0  (thread 0)
a[1] = 1  (thread 0)
a[4] = 16 (thread 2)
a[2] = 4  (thread 1)
...
```

`OMP_NUM_THREADS` の値を変えて, 現れるスレッド番号が変わることを確認せよ.



# 1. ツールの読み込み
- AIチュータ及びジョブ投入ツールの読み込み (カーネル起動後に一度実行すればよい)
  - `heytutor` : `%%hey` でAIチュータに質問できるようになる (使い方は末尾を参照)
  - `wisteria_submit` : `%%bash_submit` (先頭に `#PJM ...` を書く) でジョブ投入できるようになる


In [ ]:
import heytutor
import wisteria_submit

# 2. C++ ベースコード

In [ ]:
%%writefile_ parallel_for.cpp
#include <cstdio>
#include <omp.h>

int main() {
  const int n = 8;
  int a[n];
  // TODO: 下の for ループの直前に #pragma omp parallel for を1行追加し, 繰り返しを複数のスレッドに分担させよ.
  for (int i = 0; i < n; i++) {
    a[i] = i * i;
    printf("a[%d] = %d\t(thread %d)\n", i, a[i], omp_get_thread_num());
  }
  return 0;
}

## 2-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvc++ -fast -mp=multicore parallel_for.cpp -o parallel_for_cpp.exe

## 2-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./parallel_for_cpp.exe

# 3. Fortran ベースコード

In [ ]:
%%writefile_ parallel_for.f90
program parallel_for
  use omp_lib
  integer, parameter :: n = 8
  integer :: a(0:n-1)
  integer :: i
  ! TODO: 下の do ループを !$omp parallel do ... !$omp end parallel do で囲み, 繰り返しを複数のスレッドに分担させよ.
  do i = 0, n - 1
     a(i) = i * i
     print "(a,i0,a,i0,a,i0,a)", "a[", i, "] = ", a(i), &
          char(9)//"(thread ", omp_get_thread_num(), ")"
  end do
  ! TODO: 上で始めた parallel do 領域を閉じる (!$omp end parallel do).
end program parallel_for

## 3-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvfortran -fast -mp=multicore parallel_for.f90 -o parallel_for_f90.exe

## 3-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./parallel_for_f90.exe


# 4. AIチュータへの質問の仕方 (参考)
- 先頭で `import heytutor` 済みなら, セルに `%%hey` と書いて質問できる。
- ChatGPTなどと同様に自由に質問してよい。ただし「答えをそのまま教えて」などは自制すること。
- セル内で使える変数 (自動で展開される):
  - `{file:FILENAME}` : _FILENAME_ の中身 (例: `{file:problem.md}`, `{file:parallel_for.cpp}`)
  - `{bash[-1]}` : 最後に実行した `%%bash_` セルの入力・出力, `{bash[-2]}` = その前, ...
- 以下は質問例 (必要に応じてコピーして使う。Fortranなら `.cpp` を `.f90` に書き換える)。

## 4-1. 一般的な質問

In [ ]:
%%hey

C++の関数定義の文法どんなだっけ?

## 4-2. この問題に関するヒント

In [ ]:
%%hey

この問題に関するヒントを教えて

問題:
{file:problem.md}

## 4-3. 困ったときのヘルプ
- コンパイル時や実行時のエラー直後に実行するとエラーに関するヘルプが得られる。

In [ ]:
%%hey

以下のエラーが出た。何が間違い?

プログラム:
{file:parallel_for.cpp}

コマンドと実行結果:
{bash[-1]}

## 4-4. フィードバック
- 答えが出た後も, 無駄なところやより良いやり方がないかを聞くことを推奨。

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:parallel_for.cpp}